# OcuWeave · OTFID segmentation pilot
**Marchel Andrian Shevchenko · Data Sorcerers · 2026 · software v0.2.0**

This notebook acquires OTFID v3, audits the source archive, freezes an exploratory split, and compares an independently implemented query segmenter with an encoder/pixel-decoder matched FCN ablation. Neither network uses U-Net or pretrained weights.

This is an **image-only pilot**, not evidence of clinical validity or algorithmic superiority. Patient identities and paired narrative EMR are unverified. Active/inactive masks are unioned; the output does not predict activity, prognosis, or the origin of disease. Perceptual duplicate grouping is conservative and cannot establish patient independence.

Use **Runtime → Change runtime type → T4 GPU** if available. Runtime and Drive permissions are granted through Colab. No paid upgrade is required by this notebook. Runtime availability and quota are controlled by Google.

All raw data, audits, configurations, logs, checkpoints, and figures are saved under `MyDrive/OcuWeave — Datasets and experiments`. Raw files are preserved; data and images must not be committed to the software repository.


## 1. Connect Drive and pin the software release

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, shutil
RELEASE = 'v0.2.0'
PROJECT = Path('/content/drive/MyDrive/OcuWeave — Datasets and experiments')
PROJECT.mkdir(exist_ok=True)
REPO = Path('/content/ocuweave-' + RELEASE)
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',RELEASE,
                    'https://github.com/chenko-the-sorcerers/ocuweave.git',str(REPO)],check=True)
os.chdir(REPO)
assert subprocess.check_output(['git','describe','--tags','--exact-match'],text=True).strip() == RELEASE
REVISION = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],text=True).strip()
print('Pinned source:', RELEASE, REVISION)
# Run project modules in fresh subprocesses so Colab cannot retain stale editable imports.
def run(arguments, log=None):
    result = subprocess.run(arguments, cwd=REPO, capture_output=True, text=True)
    transcript = result.stdout + result.stderr
    if log is not None:
        Path(log).parent.mkdir(parents=True,exist_ok=True)
        Path(log).write_text(transcript)
    print(transcript[-6000:])
    result.check_returncode()
    return result
run([sys.executable,'-m','pip','install','-e','.[research]'])
import torch
print({'torch':str(torch.__version__),'cuda':torch.cuda.is_available(),
       'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
assert torch.cuda.is_available(), 'Select the approved T4 runtime before running the training pilot.'


## 2. Validate the implementation before touching real data
Tests use synthetic images. Native parity checks validate the existing C++ numerical example; it is not a differentiable training layer.

In [ ]:
LOGS = PROJECT / '04 — Runs' / RELEASE / 'validation'
LOGS.mkdir(parents=True,exist_ok=True)
run(['cmake','-S','.','-B','build','-DBUILD_TESTING=ON'],LOGS/'cmake-configure.log')
run(['cmake','--build','build'],LOGS/'cmake-build.log')
run(['ctest','--test-dir','build','--output-on-failure'],LOGS/'ctest.log')
os.environ['OCUWEAVE_NATIVE_LIBRARY'] = str(REPO/'build/libocuweave_native.so')
run([sys.executable,'-m','unittest','discover','-s','tests','-v'],LOGS/'python-tests.log')
run([sys.executable,'scripts/audit_data.py','examples/synthetic_manifest.json'],LOGS/'synthetic-audit.log')
run([sys.executable,'-m','pip','freeze'],LOGS/'environment.txt')


## 3. Acquire the official OTFID v3 archive
Source: [Zenodo record 13883485](https://zenodo.org/records/13883485), CC BY 4.0. The downloader checks the official release, license, byte count and checksum, then records SHA-256. A mismatched existing file is never overwritten. The archive is approximately 273 MB; allow additional space for runs and figures.


In [ ]:
RAW = PROJECT / '01 — Raw datasets' / 'OTFID_v3'
run([sys.executable,'scripts/download_data.py','--output',str(RAW)],LOGS/'download.log')
SOURCE_ARCHIVE = RAW / 'Ocular_Toxoplasmosis_Data_V3.zip'
print(json.loads((RAW/'download_manifest.json').read_text()))
# A disposable Colab-local copy avoids repeated reads through Drive during training.
ARCHIVE = Path('/content/Ocular_Toxoplasmosis_Data_V3.zip')
import hashlib
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
source_hash=json.loads((RAW/'download_manifest.json').read_text())['sha256']
if not ARCHIVE.exists(): shutil.copyfile(SOURCE_ARCHIVE,ARCHIVE)
assert digest(ARCHIVE)==source_hash


## 4. Audit and freeze the split
Cleaning is exclusion/reporting, not modification of source pixels. Check decoded images, mask dimensions, class/mask contradictions, missing annotations, exact pixel duplicates, and conservative dHash candidates. Exact duplicates are removed; perceptual candidates stay in the data; only pairs also passing RGB thumbnail RMSE ≤ 0.025 share a split. Empty masks are allowed only for a verified healthy label. Unknown patient IDs stay unknown.

The explicit exploratory flag permits image-group splitting. **Do not report it as a patient-disjoint benchmark.** Review `audit.json` and the QC panel before interpreting results. A new software/seed audit uses a new directory; an existing manifest is immutable.


In [ ]:
AUDIT = PROJECT / '02 — Audits' / ('otfid-v3-binary-v2-seed42-' + RELEASE)
MANIFEST = AUDIT / 'manifest.json'
if not MANIFEST.exists():
    run([sys.executable,'scripts/prepare_otfid.py','--archive',str(ARCHIVE),'--output',str(AUDIT),
         '--seed','42','--allow-unverified-patients'],LOGS/'audit.log')
else:
    manifest_data=json.loads(MANIFEST.read_text())
    assert manifest_data['archive_sha256']==source_hash and manifest_data['seed']==42
    assert manifest_data['audit_protocol']=='otfid-binary-v2'
report=json.loads((AUDIT/'audit.json').read_text())
print({key:report[key] for key in ('source_label_rows','archive_images','eligible_samples','labels','splits','split_unit')})
print('Excluded:',len(report['exclusions']),'Exact duplicates:',len(report['exact_duplicates_removed']),
      'Perceptual candidates:',len(report['near_duplicate_candidates']))
FIGURES = AUDIT / 'figures'
run([sys.executable,'scripts/plot_results.py','--manifest',str(MANIFEST),'--archive',str(ARCHIVE),
     '--output',str(FIGURES)])
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(FIGURES/'train_panel.png')))


## 5. Train the predeclared pilot
Both models use seed 42, the same split, 192-pixel letterboxing, 20 epochs, AdamW, BCE + soft Dice, and the same batch size. Letterbox padding is excluded from loss/metrics. Only validation positive-image Dice selects the checkpoint. Parameters differ because the query head adds parameters; this is an ablation with matched data/training budget, not a parameter-matched claim.

Loss logs and the best checkpoint are saved to Drive each epoch. If a session stops, keep the incomplete run for inspection and choose a new `RUN_ID`; this version does not resume optimizer state. Do not tune hyperparameters using the test set.


In [ ]:
RUN_ID = 'pilot-seed42-20epochs'
RUN_ROOT = PROJECT / '04 — Runs' / RELEASE / RUN_ID
RUN_ROOT.mkdir(parents=True,exist_ok=True)
for architecture in ('query','fcn'):
    target=RUN_ROOT/architecture
    config=REPO/'configs/experiments'/f'otfid_{architecture}_pilot.json'
    if (target/'checkpoint.json').exists():
        saved=json.loads((target/'provenance.json').read_text())
        assert saved['source_commit']==REVISION and saved['manifest_sha256']==digest(MANIFEST)
        assert saved['config']==json.loads(config.read_text())
        print('Preserving completed run:',architecture)
        continue
    assert not target.exists(), 'Incomplete run exists. Preserve it and choose a new RUN_ID.'
    run([sys.executable,'scripts/train.py','--manifest',str(MANIFEST),'--archive',str(ARCHIVE),
         '--output',str(target),'--config',str(config),'--device','cuda','--allow-unverified-patients'],
         RUN_ROOT/f'{architecture}-training.log')


## 6. Evaluate once and inspect failures
Threshold 0.5 is fixed before test evaluation. Report positive-image Dice separately from all-image Dice, because empty healthy masks can inflate the latter. Image-bootstrap intervals assume image independence and do not resolve unknown patient linkage. A single seed is a pipeline pilot; later work needs repeated seeds, independent cohorts, boundary/small-lesion analyses, and clinical annotation review.


In [ ]:
summary=[]
for architecture in ('query','fcn'):
    output=RUN_ROOT/f'{architecture}-test.json'
    if not output.exists():
        run([sys.executable,'scripts/evaluate.py','--manifest',str(MANIFEST),'--archive',str(ARCHIVE),
             '--checkpoint',str(RUN_ROOT/architecture/'best.pt'),'--output',str(output),
             '--device','cuda','--allow-unverified-patients'],RUN_ROOT/f'{architecture}-evaluation.log')
    result=json.loads(output.read_text())
    summary.append({'architecture':architecture,**result['metrics']})
print(json.dumps(summary,indent=2))
(RUN_ROOT/'comparison.json').write_text(json.dumps(summary,indent=2))
run([sys.executable,'scripts/plot_results.py','--manifest',str(MANIFEST),'--archive',str(ARCHIVE),
     '--checkpoint',str(RUN_ROOT/'query/best.pt'),'--split','test','--output',str(RUN_ROOT/'figures'),
     '--query-run',str(RUN_ROOT/'query'),'--fcn-run',str(RUN_ROOT/'fcn')])
display(DisplayImage(filename=str(RUN_ROOT/'figures/learning_curves.png')))
display(DisplayImage(filename=str(RUN_ROOT/'figures/test_panel.png')))
print('Saved to:',RUN_ROOT)


## Interpretation and next steps
Use the actual saved metrics, not these instructions, to write the result. Preserve negative findings. The first comparison cannot establish novelty or clinical benefit. Next: inspect per-image failures, verify patient linkage, collect expert review, repeat seeds, and define genuinely paired clinical experiments. No synthetic note is treated as original EMR.

**References:** O. Cardozo et al., “Dataset of fundus images for the diagnosis of ocular toxoplasmosis,” *Data in Brief*, vol. 48, Art. 109056, 2023, doi: [10.1016/j.dib.2023.109056](https://doi.org/10.1016/j.dib.2023.109056). Dataset release: [OTFID v3](https://zenodo.org/records/13883485). Architecture references and research scope: [OcuWeave project](https://data-sorcerers.com/ocuweave/).
